# Text preprocessing pipeline

In [ ]:
#Imports and reseting the environment

%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import nltk, pandas as pd, numpy.testing as npt, unicodedata, contractions, re
from numpy.testing import assert_equal as eq
import unittest
from colorunittest import run_unittest
_ = nltk.download(['omw-1.4','brown','wordnet','stopwords','averaged_perceptron_tagger'], quiet=True)
from nltk.corpus import brown, stopwords
from nltk.corpus.reader.wordnet import NOUN, VERB, ADJ

In [ ]:
class Pipe():
    #Attribute Initialization
    def __init__(self, LsWords=[], SsLex=set(), SsStopWords=set()) -> object:
        assert isinstance(LsWords, list) or LsWords is None, f'LsWords must be a list, not a {type(LsWords)}'
        assert isinstance(SsLex, set) or (SsLex=='nltk'), f'SsLex must be "nltk" or a set of lexicon words, not a {type(SsLex)}'
        assert isinstance(SsStopWords, set) or (SsStopWords=='nltk'), f'SsStopWords must be "nltk" or a set of words, not a {type(SsStopWords)}'
        self.df = pd.DataFrame(columns = ['Step', 'Words', 'Vocab', 'CorrVocab'])

        _ = nltk.download(['brown'], quiet=True)
        Ss6 = {s.lower() for s in nltk.corpus.brown.words()}

        self.LsWords = LsWords
        if SsLex =='nltk':
            self.SsLex = Ss6
        else:
            self.SsLex = SsLex
        if SsStopWords =='nltk':
            self.SsStopWords = set(stopwords.words('english'))
        else:
            self.SsStopWords = SsStopWords

        self.AddStats('Initialize')     # Saves basic stats for LsWord

    ### Step 0: Output
    def Out(self) -> str:
        cleaned_string = ' '.join(self.LsWords)
        cleaned_string = re.sub(r'\s+', ' ', cleaned_string)
        return cleaned_string.strip()

    ### Step 1: Lowercase
    @property
    def Low(self) -> object:
        self.LsWords = [w.lower() for w in self.LsWords]  # Lowercase each word token
        self.AddStats('Lower')  # Update statistics for this step **** not sure about this
        return self  # Return reference to self for method chaining
        raise NotImplementedError()

    ### Step 2: Remove Digits
    @property
    def NoNum(self) -> object:
        self.LsWords = [re.sub(r'\d+', '', word) for word in self.LsWords]  # Remove all digits
        self.AddStats('NoNum')  # Update statistics for this step
        return self
        raise NotImplementedError()

    ### Step 3: Keep Only Word Characters
    @property
    def Words(self) -> object:
        self.LsWords = [re.sub(r'[^\w\s]+', '', word) for word in self.LsWords]
        self.AddStats('Words')
        return self
        raise NotImplementedError()

    ### Step 4: Remove Stop Words
    @property
    def Stop(self) -> object:
        self.LsWords = [word for word in self.LsWords if word.lower() not in self.SsStopWords]
        self.AddStats('Stop')
        return self
        raise NotImplementedError()

    ### Step 5: Normalization
    @property
    def Norm(self) -> object:
        self.LsWords = [unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8') for word in self.LsWords]
        self.AddStats('Norm')  # Update statistics for this step
        return self  # Return reference to self for method chaining
        raise NotImplementedError()

    ### Step 5: Expand Contractions
    @property
    def Exp(self) -> object:
        concatenated_string = ' '.join(self.LsWords)
        expanded_string = contractions.fix(concatenated_string)
        self.LsWords = expanded_string.split()
        self.AddStats('Exp')  # Update statistics for this step
        return self  # Return reference to self for method chaining
        raise NotImplementedError()


    ### TASK 9: Stem
    @property
    def Stem(self) -> object:
        pso = nltk.stem.PorterStemmer()       # instantiates Porter Stemmer object
        self.LsWords = [pso.stem(word) for word in self.LsWords]
        self.AddStats('Stem')
        return self
        raise NotImplementedError()

    ### TASK 10: Lemmatize
    @property
    def Lem(self) -> object:
        wlo = nltk.stem.WordNetLemmatizer()   # instantiates WordNet Lemmatizer object
        WNTag = lambda t: t[0].lower() if t[0] in 'ARNV' else 'n'   # Converts NLTK POS Tag to WordNet POS Tag
        # Create a list of tuples of words & their WordNet POS tags,
        #    i.e. 'a' for adjectives, 'r' for adverbs, 'v' for verbs, 'n' for nouns and all else
        LTssWordTag = [(word, WNTag(tag)) for word, tag in nltk.pos_tag(self.LsWords)]
        WNTag = lambda t: t[0].lower() if t[0] in 'ARNV' else 'n'
        LTssWordTag = [(word, WNTag(tag)) for word, tag in nltk.pos_tag(self.LsWords)]
        self.LsWords = [wlo.lemmatize(word, tag) for word, tag in LTssWordTag]
        self.AddStats('Lem')  # Update statistics for this step
        return self
        raise NotImplementedError()

    def AddStats(self, sTask='') -> object:
        SsWords = {s for s in self.LsWords}
        self.df.loc[len(self.df)] = [sTask, len(self.LsWords), len(SsWords), len(SsWords.intersection(self.SsLex))]
        return self     # Finally, return reference to the object itself

In [ ]:
#download sample text
_ = nltk.download(['gutenberg'], quiet=True)
LsBookWords = list(nltk.corpus.gutenberg.words('bryant-stories.txt')) #[:1000]
sSampleText = nltk.corpus.gutenberg.raw('bryant-stories.txt')[:500] + '...\n'
print(sSampleText)

[Stories to Tell to Children by Sara Cone Bryant 1918] 


TWO LITTLE RIDDLES IN RHYME


     There's a garden that I ken,
     Full of little gentlemen;
     Little caps of blue they wear,
     And green ribbons, very fair.
           (Flax.)

     From house to house he goes,
     A messenger small and slight,
     And whether it rains or snows,
     He sleeps outside in the night.
           (The path.)




THE LITTLE YELLOW TULIP


Once there was a little yellow Tulip,...



In [ ]:
#print sample text lowercase (step 1)

low = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Low
low.Out()[:500]

"[ stories to tell to children by sara cone bryant 1918 ] two little riddles in rhyme there ' s a garden that i ken , full of little gentlemen ; little caps of blue they wear , and green ribbons , very fair . ( flax .) from house to house he goes , a messenger small and slight , and whether it rains or snows , he sleeps outside in the night . ( the path .) the little yellow tulip once there was a little yellow tulip , and she lived down in a little dark house under the ground . one day she was si"

In [ ]:
#print sample text with no numbers (step 2)

norm = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').NoNum
norm.Out()[:500]

"[ Stories to Tell to Children by Sara Cone Bryant ] TWO LITTLE RIDDLES IN RHYME There ' s a garden that I ken , Full of little gentlemen ; Little caps of blue they wear , And green ribbons , very fair . ( Flax .) From house to house he goes , A messenger small and slight , And whether it rains or snows , He sleeps outside in the night . ( The path .) THE LITTLE YELLOW TULIP Once there was a little yellow Tulip , and she lived down in a little dark house under the ground . One day she was sitting"

In [ ]:
#print sample text with no numbers (step 3)

words = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Words
words.Out()[:500]

'Stories to Tell to Children by Sara Cone Bryant 1918 TWO LITTLE RIDDLES IN RHYME There s a garden that I ken Full of little gentlemen Little caps of blue they wear And green ribbons very fair Flax From house to house he goes A messenger small and slight And whether it rains or snows He sleeps outside in the night The path THE LITTLE YELLOW TULIP Once there was a little yellow Tulip and she lived down in a little dark house under the ground One day she was sitting there all by herself and it was '

In [ ]:
#print sample text with no stop words (step 4)

stop = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Stop
stop.Out()[:500]

'[ Stories Tell Children Sara Cone Bryant 1918 ] TWO LITTLE RIDDLES RHYME \' garden ken , Full little gentlemen ; Little caps blue wear , green ribbons , fair . ( Flax .) house house goes , messenger small slight , whether rains snows , sleeps outside night . ( path .) LITTLE YELLOW TULIP little yellow Tulip , lived little dark house ground . One day sitting , , still . Suddenly , heard little _tap , tap , tap_ , door . " ?" said . " \' Rain , want come ," said soft , sad , little voice . " , \' com'

In [ ]:
#print sample text with no stop words (step 5)

norm = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Norm
norm.Out()[:500]

"[ Stories to Tell to Children by Sara Cone Bryant 1918 ] TWO LITTLE RIDDLES IN RHYME There ' s a garden that I ken , Full of little gentlemen ; Little caps of blue they wear , And green ribbons , very fair . ( Flax .) From house to house he goes , A messenger small and slight , And whether it rains or snows , He sleeps outside in the night . ( The path .) THE LITTLE YELLOW TULIP Once there was a little yellow Tulip , and she lived down in a little dark house under the ground . One day she was si"

In [ ]:
#print sample text with no stop words (step 6)

test = ["can't", "cat", "won't", "There's"]
exp = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Exp
exp.Out()[:500]

#need to check on there's

"[ Stories to Tell to Children by Sara Cone Bryant 1918 ] TWO LITTLE RIDDLES IN RHYME There ' s a garden that I ken , Full of little gentlemen ; Little caps of blue they wear , And green ribbons , very fair . ( Flax .) From house to house he goes , A messenger small and slight , And whether it rains or snows , He sleeps outside in the night . ( The path .) THE LITTLE YELLOW TULIP Once there was a little yellow Tulip , and she lived down in a little dark house under the ground . One day she was si"

In [ ]:
#print sample text with no stop words (step 7)

test = ["can't", "cat", "won't", "There's"]
stem = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Stem
stem.Out()[:500]

"[ stori to tell to children by sara cone bryant 1918 ] two littl riddl IN rhyme there ' s a garden that I ken , full of littl gentlemen ; littl cap of blue they wear , and green ribbon , veri fair . ( flax .) from hous to hous he goe , A messeng small and slight , and whether it rain or snow , He sleep outsid in the night . ( the path .) the littl yellow tulip onc there wa a littl yellow tulip , and she live down in a littl dark hous under the ground . one day she wa sit there , all by herself ,"

In [ ]:
#print sample text with no stop words (step 8)

lem = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Lem
lem.Out()[:500]

"[ Stories to Tell to Children by Sara Cone Bryant 1918 ] TWO LITTLE RIDDLES IN RHYME There ' s a garden that I ken , Full of little gentleman ; Little cap of blue they wear , And green ribbon , very fair . ( Flax .) From house to house he go , A messenger small and slight , And whether it rain or snow , He sleep outside in the night . ( The path .) THE LITTLE YELLOW TULIP Once there be a little yellow Tulip , and she live down in a little dark house under the ground . One day she be sit there , "

In [ ]:
#example of stringing several of the process togeather

pp = Pipe(LsBookWords, SsStopWords='nltk', SsLex='nltk').Low.Norm.Exp.Words.Stem.Stop.NoNum
pp.Out()[:500]
pp.df

'stori tell children sara cone bryant two littl riddl rhyme garden ken full littl gentlemen littl cap blue wear green ribbon veri fair flax hous hous goe messeng small slight whether rain snow sleep outsid night path littl yellow tulip onc wa littl yellow tulip live littl dark hous ground one day wa sit wa veri still suddenli heard littl _tap tap tap_ door said rain want come said soft sad littl voic come littl tulip said heard anoth littl _tap tap tap_ window pane said soft littl voic answer rai'

,Step,Words,Vocab,CorrVocab
0,Initialize,55563,4420,3307
1,Lower,55563,3940,3448
2,Norm,55563,3940,3448
3,Exp,55572,3935,3445
4,Words,55572,3896,3432
5,Stem,55572,2998,1955
6,Stop,32561,2882,1848
7,NoNum,32561,2880,1846


"[ stories to tell to children by sara cone bryant 1918 ] two little riddles in rhyme there ' s a garden that i ken , full of little gentlemen ; little caps of blue they wear , and green ribbons , very fair . ( flax .) from house to house he goes , a messenger small and slight , and whether it rains or snows , he sleeps outside in the night . ( the path .) the little yellow tulip once there was a little yellow tulip , and she lived down in a little dark house under the ground . one day she was si"